# 🌌 SWAYAMBHU: Sovereign Integrated Intelligence v1(Kaggle Brain)


In [11]:
!pip install fastapi uvicorn groq firebase-admin pyngrok nest_asyncio httpx

In [ ]:
import os, json, time, threading, asyncio, re, base64
import nest_asyncio
import firebase_admin
from firebase_admin import credentials, firestore
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok, conf
from groq import Groq
import httpx
import uvicorn

# 1. Fix Kaggle's async loop
nest_asyncio.apply()

# 2. Setup App & Secrets
PORT = 8000
app = FastAPI(title="Swayambhu Cloud Gateway")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

# 🚀 ARCHITECTURAL FIX 1: Unified Health Endpoint for Mac Watchdog
@app.get("/health")
async def health_endpoint():
    return {"status": "ok", "brain": "online", "defcon": 5}
    
db = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for k in ["GROQ_API_KEY", "NGROK_TOKEN", "ELEVENLABS_API_KEY", "ELEVENLABS_VOICE_ID", "FIREBASE_B64"]:
        try: os.environ[k] = user_secrets.get_secret(k)
        except: pass
    
    # Init Firebase using the Base64 secret stored in Kaggle
    b64_cred = os.environ.get("FIREBASE_B64")
    if b64_cred:
        cred_dict = json.loads(base64.b64decode(b64_cred).decode())
        if not firebase_admin._apps:
            firebase_admin.initialize_app(credentials.Certificate(cred_dict))
        db = firestore.client(database_id="ai-studio-47a94c34-82ae-4dc7-af57-09fb23251026")
except Exception as e:
    print(f"⚠️ Secret loading error: {e}")

groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY", ""))

def get_dynamic_temperature(prompt: str) -> float:
    """The 'Two-Brain' Toggle: Sets hallucination allowance based on intent."""
    creative_keywords = ['idea', 'brainstorm', 'creative', 'draft', 'imagine', 'story', 'write', 'conceptualize', 'design']
    if any(word in prompt.lower() for word in creative_keywords):
        return 0.8 
    return 0.1

# ── 3. The Commander (70B) Unified Endpoints ──
async def _process_json_intent(payload: dict):
    command = payload.get("command", "")
    
    # 1. Grab the dynamic "Big Picture" prompt generated by your Mac Body
    sys_override = payload.get("sys_override", "")
    
    if not command:
        return {"message": "Empty command received.", "plan": []}

    system_content = sys_override if sys_override else "You are Swayambhu. Respond in valid JSON."
    
    # 2. Surgically inject the execution rules without breaking the schema
    system_content += (
        "\n\nCRITICAL EXECUTION RULES:\n"
        "1. You MUST respond in valid JSON format.\n"
        "2. When using 'execute_mac_command', ALWAYS prefer robust shell commands (Zsh/Bash) like `open -a AppName` or `open \"URL\"`.\n"
        "3. NEVER write fragile AppleScript (e.g., `tell window 1 to set URL`) for simple app navigation or web searches."
    )

    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": command}
    ]

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            response_format={"type": "json_object"},
            temperature=0.1 
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        return {"message": f"Brain routing failure: {str(e)}", "plan": []}
@app.post("/edge_command")
async def handle_edge_command(payload: dict):
    """Primary endpoint for live Mac commands."""
    return await _process_json_intent(payload)

@app.post("/command")
async def handle_legacy_command(payload: dict):
    """Fallback endpoint for AirGapSurvivalMode offline queue flushes."""
    return await _process_json_intent(payload)


# ── 4. Neural Pipeline (WebSocket + TTS) ──
_ELEVENLABS_KEY = os.environ.get("ELEVENLABS_API_KEY", "")
_ELEVENLABS_VOICE = os.environ.get("ELEVENLABS_VOICE_ID", "21m00Tcm4TlvDq8ikWAM")

async def elevenlabs_tts_bytes(text: str):
    if not _ELEVENLABS_KEY or not text.strip(): return None
    url = f"https://api.elevenlabs.io/v1/text-to-speech/{_ELEVENLABS_VOICE}/stream"
    headers = {"xi-api-key": _ELEVENLABS_KEY, "Content-Type": "application/json"}
    body = {"text": text.strip(), "model_id": "eleven_turbo_v2", "optimize_streaming_latency": 4}
    try:
        async with httpx.AsyncClient() as client:
            async with client.stream("POST", url, headers=headers, json=body) as resp:
                if resp.status_code == 200:
                    chunks = [chunk async for chunk in resp.aiter_bytes(4096)]
                    return b"".join(chunks)
    except Exception: return None

@app.websocket("/ws_stream")
async def ws_stream_endpoint(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            raw = await websocket.receive_text()
            msg = json.loads(raw)
            prompt = msg.get("prompt", "")
            if not prompt: continue
            
            stream = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                stream=True
            )
            sentence_buf = ""
            for chunk in stream:
                delta = chunk.choices[0].delta.content
                if delta:
                    await websocket.send_json({"type": "token", "text": delta})
                    sentence_buf += delta
                    if bool(sentence_buf) and sentence_buf[-1] in ".!?\n":
                        mp3 = await elevenlabs_tts_bytes(sentence_buf)
                        if mp3:
                            await websocket.send_json({"type": "audio_start"})
                            await websocket.send_bytes(mp3)
                        sentence_buf = ""
            await websocket.send_json({"type": "done"})
    except WebSocketDisconnect:
        pass

# ── 5. Speculative Relay ──
@app.post("/speculative")
async def speculative_endpoint(request: dict):
    prompt = request.get("prompt", "")
    draft = request.get("draft", "")
    max_tokens = request.get("max_tokens", 400)
    
    messages = [
        {"role": "system", "content": "You are Swayambhu, a Sovereign AI. Respond accurately and concisely."},
        {"role": "user", "content": prompt}
    ]
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            max_tokens=max_tokens,
            temperature=0.2
        )
        final_text = response.choices[0].message.content.strip()
        draft_words = draft.split()
        final_words = final_text.split()
        
        accepted = 0
        for d_word, f_word in zip(draft_words, final_words):
            if d_word.lower() == f_word.lower(): accepted += 1
            else: break
                
        return {
            "text": final_text,
            "source": "cloud_verified",
            "accepted_tokens": accepted,
            "rejected_tokens": max(0, len(draft_words) - accepted)
        }
    except Exception as e:
        return {"text": "", "source": f"cloud_error: {str(e)}", "accepted_tokens": 0, "rejected_tokens": 0}

# ── 6. Web3 Settlement ──
class Web3SettlementLayer:
    def __init__(self, network="Polygon"):
        self.network = network
        self.public_address = "0xSwayambhuSovereignNode00000000000000000"
    def generate_x402_invoice(self, client_id, amount_usdc, task_desc):
        invoice_id = f"INV-{int(time.time())}"
        return {"invoice_id": invoice_id, "pay_to": self.public_address, "amount_usdc": amount_usdc, "network": self.network, "memo": task_desc, "status": "AWAITING_FUNDS"}

crypto_layer = Web3SettlementLayer()

@app.post("/sandbox/evaluate")
def b2b_sandbox_service(agent_payload: dict):
    client_id = agent_payload.get("client_id", "Unknown")
    invoice = crypto_layer.generate_x402_invoice(client_id, 5.0, "Agent Stress Test")
    return {"status": "EVALUATION_COMPLETE", "vulnerability_score": "HIGH", "failures_detected": ["Prompt Injection Vulnerability detected"], "invoice": invoice}

# ── 7. Boot Ngrok & Publish to Firebase ──
def _start_ngrok():
    ngrok.kill()
    conf.get_default().auth_token = os.environ.get("NGROK_TOKEN", "")
    tunnel = ngrok.connect(PORT)
    url = tunnel.public_url
    print(f"\n🚀 SWAYAMBHU CLOUD ONLINE AT: {url}\n")
    
    if db:
        try:
            db.document('artifacts/SWAYAMBHU_SOVEREIGN_001/public/data').set({
                'brain_url': url,
                'updated_at': firestore.SERVER_TIMESTAMP,
                'status': 'ONLINE'
            }, merge=True)
            print("🔥 Firebase updated with new Brain URL.")
        except Exception as e:
            print(f"⚠️ Firebase URL sync failed: {e}")

threading.Thread(target=_start_ngrok, daemon=True).start()
time.sleep(2)

# ── 8. Safe Async Boot ──
config = uvicorn.Config(app, host='0.0.0.0', port=PORT, log_level='warning')
server = uvicorn.Server(config)
await server.serve()


🚀 SWAYAMBHU CLOUD ONLINE AT: https://dayton-backdoor-celestially.ngrok-free.dev

🔥 Firebase updated with new Brain URL.
